# GPU callback-fusion benchmark (PR #125)

Same-runtime A/B benchmark for [issue #124](https://github.com/JBjoernskov/Twin4Build/issues/124):

- **baseline**: current `dev`
- **fused**: `feature/issue-124/fuse-collocation-callbacks`

Each arm runs in a fresh Python subprocess from its own shallow clone, on the same Colab GPU. This avoids module-cache contamination while preserving the same hardware, PyTorch, CUDA, data, optimizer settings, and process startup conditions.

The benchmark reports wall time, IPOPT status/iterations/objective, maximum continuity defect, peak CUDA memory, per-sensor RMSE, and—on the fused arm—physical forward/derivative evaluations and cache hits.

> Select **Runtime → Change runtime type → GPU** before running. Default workload: 5 days, 20-minute steps, exact Hessian, 100 IPOPT iterations. Both arms may hit the iteration limit; compare optimization mechanics and intermediate feasibility, not final calibration quality, unless both genuinely converge.

In [ ]:
# --- Setup -----------------------------------------------------------------
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys

import torch

REPO_URL = "https://github.com/JBjoernskov/Twin4Build.git"
BASE_REF = "dev"
CANDIDATE_REF = "feature/issue-124/fuse-collocation-callbacks"
ROOT = Path("/content/twin4build_callback_benchmark")
BASE_DIR = ROOT / "baseline"
CANDIDATE_DIR = ROOT / "candidate"

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA Colab runtime is required for this benchmark.")

# Install dependencies and the candidate package using the repository's normal
# git-ref flow. The arm subprocesses override PYTHONPATH with their own clone.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        f"git+{REPO_URL}@{CANDIDATE_REF}",
    ],
    check=True,
)

shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True)
for ref, destination in ((BASE_REF, BASE_DIR), (CANDIDATE_REF, CANDIDATE_DIR)):
    subprocess.run(
        [
            "git",
            "clone",
            "--quiet",
            "--depth",
            "1",
            "--branch",
            ref,
            REPO_URL,
            str(destination),
        ],
        check=True,
    )

candidate_source = (CANDIDATE_DIR / "twin4build/estimator/_transcription.py").read_text()
required = ["class _IterateCache", "_assemble_objective_gradient", '"callback_cache"']
missing = [symbol for symbol in required if symbol not in candidate_source]
if missing:
    raise RuntimeError(f"Candidate clone is stale; missing {missing}")

props = torch.cuda.get_device_properties(0)
HARDWARE = {
    "gpu": props.name,
    "gpu_memory_gb": props.total_memory / 1e9,
    "torch": torch.__version__,
    "python": platform.python_version(),
}
print(json.dumps(HARDWARE, indent=2))

In [ ]:
# --- Write the isolated arm runner -----------------------------------------
ARM_RUNNER = ROOT / "run_callback_arm.py"
ARM_RUNNER.write_text(r'''
import datetime
import json
import os
from pathlib import Path
import subprocess
import sys
import time

import torch

repo = Path(sys.argv[1]).resolve()
label = sys.argv[2]
out_file = Path(sys.argv[3]).resolve()
hours = int(os.environ.get("T4B_BENCH_HOURS", "120"))
maxiter = int(os.environ.get("T4B_BENCH_MAXITER", "100"))

# Force this process to import the selected checkout, not the package installed
# by the setup cell.
sys.path.insert(0, str(repo))
os.chdir(repo)

import importlib.util

from dateutil import tz
import twin4build as tb
import twin4build.estimator._casadi_ipopt as ipopt
import twin4build.examples as examples_package
import twin4build.examples.utils as example_utils

tb._IS_TESTING = True

# Use the exact full-workflow model and parameterization that completed all 100
# iterations in the earlier A100 Hessian benchmark. The simpler
# collocation_comparison fixture can produce invalid derivatives at its initial
# point and is not a valid end-to-end benchmark for this optimization.
example_path = Path(examples_package.__file__).parent / "full_workflow_example.py"
spec = importlib.util.spec_from_file_location("_callback_benchmark_workflow", example_path)
workflow = importlib.util.module_from_spec(spec)
spec.loader.exec_module(workflow)

STEP_SIZE = 1200
START = datetime.datetime(2023, 12, 2, tzinfo=tz.gettz("Europe/Copenhagen"))


def build_model():
    model = tb.Model(id=f"callback_{label}")
    model.load(
        semantic_model_filename=example_utils.get_path(
            ["estimator_example", "one_room_example_model.xlsm"]
        ),
        fcn=workflow.fcn,
    )
    model.to("cuda", torch.float64)
    return model


def build_parameters(model):
    c = model.components
    space, heater = c["office"], c["office_space_heater"]
    hc = c["office_temperature_heating_controller"]
    cc = c["office_co2_controller"]
    valve = c["office_space_heater_valve"]
    sup, exh = c["office_supply_damper"], c["office_exhaust_damper"]
    occ, wall = c["office_occupancy"], c["office_boundary_wall"]
    det = c["office_occupancy_detector"]
    return [
        (space, "thermal.C_air", 5e5, 1e4, 5e5),
        (space, "thermal.C_wall", 1e6, 1e5, 3e6),
        (wall, "C", 1e6, 1e4, 1e7),
        (space, "thermal.R_out", 0.5, 0.01, 1),
        (space, "thermal.R_in", 0.1, 0.01, 1),
        (wall, "R_a", 0.04, 1e-4, 1),
        (wall, "R_b", 0.04, 1e-4, 1),
        (space, "thermal.f_wall", 0.1, 0, 10),
        (space, "thermal.f_air", 0.1, 0, 10),
        (space, "thermal.Q_occ_gain", 100.0, 10, 200),
        (heater, "thermalMassHeatCapacity", 1e4, 1e3, 2e5),
        (heater, "UA", None, 1, 100),
        (hc, "kp", 0.005, 1e-5, 1, "private"),
        (cc, "kp", 0.0001, 1e-5, 1, "private"),
        ([hc, cc], "Ti", 30, 1, 300, "private"),
        ([hc, cc], "Td", 0, 0, 1, "private"),
        (valve, "waterFlowRateMax", 0.001, 1e-6, 0.1),
        (valve, "valveAuthority", 1, 0.4, 1),
        ([sup, occ.supply_damper], "a", 1, 1, 10, "shared"),
        ([sup, occ.supply_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([exh, occ.exhaust_damper], "a", 1, 1, 10, "shared"),
        ([exh, occ.exhaust_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([space, occ], "mass.V", 65, 50, 80, "shared"),
        ([space, occ], "mass.G_occ", 1e-6, 1e-6, 1e-5, "shared"),
        ([space, occ], "mass.m_inf", 0.001, 1e-4, 0.01, "shared"),
        (det, "threshold", 1.0, 0.02, 5.0),
    ]


def build_measurements(model):
    c = model.components
    return [
        (c["office_valve_position_sensor"], 0.05 / 2),
        (c["office_temperature_sensor"], 0.1 / 2),
        (c["office_damper_position_sensor"], 0.05 / 2),
        (c["office_co2_sensor"], 30 / 2),
    ]


solver_meta = {}
callback_stats = {
    name: {"calls": 0, "seconds": 0.0}
    for name in ("objective", "gradient", "constraints", "jacobian", "hessian")
}
original_solve = ipopt.solve_ipopt_constrained


def timed_callback(name, callback):
    if callback is None:
        return None

    def wrapped(*args):
        started = time.perf_counter()
        value = callback(*args)
        callback_stats[name]["calls"] += 1
        callback_stats[name]["seconds"] += time.perf_counter() - started
        return value

    return wrapped


def measured_solve(
    x0, lb, ub, fun, grad, n_g, g_fun, g_jac_vals, jac_rows, jac_cols,
    options=None, *, hess_vals=None, hess_rows=None, hess_cols=None,
    early_stopping=None, print_level=0, quiet=True,
):
    torch.cuda.synchronize()
    started = time.perf_counter()
    result = original_solve(
        x0,
        lb,
        ub,
        timed_callback("objective", fun),
        timed_callback("gradient", grad),
        n_g,
        timed_callback("constraints", g_fun),
        timed_callback("jacobian", g_jac_vals),
        jac_rows,
        jac_cols,
        options=options,
        hess_vals=timed_callback("hessian", hess_vals),
        hess_rows=hess_rows,
        hess_cols=hess_cols,
        early_stopping=early_stopping,
        print_level=print_level,
        quiet=quiet,
    )
    torch.cuda.synchronize()
    solver_meta.update(
        seconds=time.perf_counter() - started,
        status=str(result.status),
        success=bool(result.success),
        iterations=None if result.nit is None else int(result.nit),
        objective=float(result.fun),
        callbacks=callback_stats,
    )
    return result


ipopt.solve_ipopt_constrained = measured_solve
model = build_model()
estimator = tb.Estimator(tb.Simulator(model))
end = START + datetime.timedelta(hours=hours)

# Measure the complete estimate separately from IPOPT's solver-only interval.
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
started = time.perf_counter()
result = estimator.estimate(
    parameters=build_parameters(model),
    measurements=build_measurements(model),
    start_time=[START],
    end_time=[end],
    step_size=STEP_SIZE,
    n_warmup=20,
    method=("casadi", "ipopt", "ad", "collocation"),
    options={
        "maxiter": maxiter,
        "exact_hessian": True,
        "early_stopping": False,
        "boundary_state_init": "rollout",
    },
)
torch.cuda.synchronize()
estimate_seconds = time.perf_counter() - started
if solver_meta.get("iterations", 0) == 0 or solver_meta.get("status") == "Invalid_Number_Detected":
    raise RuntimeError(
        f"{label} did not enter the optimization loop: {solver_meta}. "
        "This is not a valid benchmark result."
    )
audit = result.get("transcription_audit", {})
row = {
    "arm": label,
    "ref": subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"], cwd=repo, text=True
    ).strip(),
    "hours": hours,
    "maxiter": maxiter,
    "estimate_seconds": estimate_seconds,
    "solver_seconds": solver_meta.get("seconds"),
    "status": solver_meta.get("status", audit.get("return_status")),
    "success": solver_meta.get("success"),
    "iterations": solver_meta.get("iterations"),
    "objective": solver_meta.get("objective"),
    "max_defect": audit.get("max_defect"),
    "peak_cuda_memory_gb": torch.cuda.max_memory_allocated() / 1e9,
    "callback_timings": solver_meta.get("callbacks", {}),
    "callback_cache": audit.get("callback_cache", {}),
    "sensor_rmse": {
        sensor: values["nlp_rmse"]
        for sensor, values in audit.get("per_sensor", {}).items()
    },
}
out_file.write_text(json.dumps(row, indent=2))
print(json.dumps(row, indent=2))
''')
print(f"wrote {ARM_RUNNER}")

In [ ]:
# --- Run both arms ---------------------------------------------------------
# Reduce these for a smoke test; restore 120/100 for the comparable A100 run.
BENCH_HOURS = 120
BENCH_MAXITER = 100

result_files = {
    "baseline_dev": ROOT / "baseline_result.json",
    "fused_pr125": ROOT / "candidate_result.json",
}
arm_dirs = {
    "baseline_dev": BASE_DIR,
    "fused_pr125": CANDIDATE_DIR,
}

for label, repo_dir in arm_dirs.items():
    print(f"\nRunning {label} from {repo_dir} ...", flush=True)
    env = os.environ.copy()
    env["T4B_BENCH_HOURS"] = str(BENCH_HOURS)
    env["T4B_BENCH_MAXITER"] = str(BENCH_MAXITER)
    env["PYTHONPATH"] = str(repo_dir) + os.pathsep + env.get("PYTHONPATH", "")
    subprocess.run(
        [sys.executable, str(ARM_RUNNER), str(repo_dir), label, str(result_files[label])],
        cwd=repo_dir,
        env=env,
        check=True,
    )

rows = [json.loads(result_files[label].read_text()) for label in arm_dirs]
print("\nBoth arms completed.")

## Results

The decisive comparison is **baseline `dev` vs fused PR #125**. Use `solver_seconds` for the optimization-path speedup and `callback_total_seconds` to isolate Python/Torch callback work. The old objective callback eagerly computes value+gradient, so its cost appears mostly under `objective`; the fused arm moves derivative work to `gradient` and can reuse it from `jacobian`/Hessian requests.

Only treat objective/RMSE differences as meaningful if both arms have compatible convergence status and defects. Tiny arithmetic reordering can change IPOPT's path even when callbacks are mathematically equivalent.

In [ ]:
# --- Comparison tables -----------------------------------------------------
import pandas as pd

summary_rows = []
for row in rows:
    flat = {
        key: row.get(key)
        for key in (
            "arm",
            "ref",
            "hours",
            "maxiter",
            "estimate_seconds",
            "solver_seconds",
            "status",
            "success",
            "iterations",
            "objective",
            "max_defect",
            "peak_cuda_memory_gb",
        )
    }
    callback_total = 0.0
    for name, values in row.get("callback_timings", {}).items():
        flat[f"{name}_calls"] = values["calls"]
        flat[f"{name}_seconds"] = values["seconds"]
        callback_total += values["seconds"]
    flat["callback_total_seconds"] = callback_total
    for name, value in row.get("callback_cache", {}).items():
        flat[name] = value
    summary_rows.append(flat)

summary = pd.DataFrame(summary_rows)
baseline = summary.loc[summary.arm == "baseline_dev"].iloc[0]
fused = summary.loc[summary.arm == "fused_pr125"].iloc[0]
summary["solver_speedup_vs_old"] = baseline.solver_seconds / summary.solver_seconds
summary["callback_speedup_vs_old"] = (
    baseline.callback_total_seconds / summary.callback_total_seconds
)

pd.set_option("display.max_columns", None)
display(summary)

rmse = pd.DataFrame(
    [
        {"arm": row["arm"], **row.get("sensor_rmse", {})}
        for row in rows
    ]
)
display(rmse)

print(
    f"PR #125 solver speedup vs old dev: "
    f"{baseline.solver_seconds / fused.solver_seconds:.3f}x"
)
print(
    f"PR #125 total callback speedup vs old dev: "
    f"{baseline.callback_total_seconds / fused.callback_total_seconds:.3f}x"
)
print("\nFused physical evaluation/cache counts:")
print(json.dumps(rows[1].get("callback_cache", {}), indent=2))

if baseline.status != fused.status:
    print("\nWARNING: solver statuses differ; do not interpret objective/RMSE as final-fit parity.")